# module-base-class-custom — ex2: recursive train()/eval() toggle propagates .training to every submodule

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `module-base-class-custom`. Running the final beacon cell reports progress against the `Backprop: Module base class custom` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Module base class custom` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-base-class-custom`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-base-class-custom"
DD_SUBTOPIC = "Backprop: Module base class custom"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Module.train() / eval() — recursive toggle — deepening

Beyond the `__setattr__`-as-registrar pattern, a Module base class needs ONE more piece: a recursive `.train(mode)` / `.eval()` toggle that flips a `.training` flag on EVERY submodule.

```python
class Module:
    def __init__(self):
        ...
        object.__setattr__(self, 'training', True)
    def train(self, mode: bool = True):
        self.training = mode
        for m in self._modules.values():
            m.train(mode)
        return self
    def eval(self):
        return self.train(False)
```

**Why every submodule.** BatchNorm and Dropout in nested layers read `self.training` to decide their forward behavior. A single `model.eval()` call must reach every leaf — otherwise a deeply-nested BN keeps updating running stats during inference. Bug.

**Returns `self`** so you can chain: `model.eval().to(device)`.

### Exercise 2 — recursive train()/eval() toggle propagates .training to every submodule

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the recursive `.train(mode)` pattern over a custom Module base class so that toggling the root module's training state propagates to every directly-assigned submodule and transitively.
> Keywords: module, train-mode, eval-mode, recursive, training-flag
> ```

**KCs targeted:** `module-base-class-custom`, `module-train-eval-toggle`

Extend the minimal `Module` base class from ex1 with a `.training` flag and `.train(mode)` / `.eval()` methods that recursively toggle every submodule.

Required surface (re-define both `Parameter` and `Module` here so the drill is self-contained):

**1. `Parameter(MiniTensor)`** — same as ex1.

**2. `Module` base class with:**
   - `__init__(self)` — initializes `_parameters = {}`, `_modules = {}`, AND `training = True` (all via `object.__setattr__` to bypass the custom `__setattr__`).
   - `__setattr__` — same registrar pattern as ex1.
   - `parameters(self)` — same recursive walker as ex1.
   - **`train(self, mode=True)`** — set `self.training = mode`, then call `m.train(mode)` for every `m in self._modules.values()`. Return `self`.
   - **`eval(self)`** — call `self.train(False)` and return its result (= self).
   - `forward(self, *args, **kwargs)` — raise NotImplementedError.

**Don't forget to bootstrap `training` in `__init__`.** Using `self.training = True` would route through the custom `__setattr__`, which expects `_parameters` to already exist. Use `object.__setattr__(self, 'training', True)`.

**Don't forget to return self.** Both `train()` and `eval()` must return `self` so the chaining idiom `model.eval()(x)` works.

In [ ]:
class Parameter(MiniTensor):
    def __init__(self, array, requires_grad: bool = True):
        super().__init__(array, requires_grad=requires_grad)


class Module:
    """Module with recursive train()/eval() toggle."""
    def __init__(self):
        raise NotImplementedError()

    def __setattr__(self, name, value):
        raise NotImplementedError()

    def parameters(self):
        raise NotImplementedError()

    def train(self, mode: bool = True):
        raise NotImplementedError()

    def eval(self):
        raise NotImplementedError()

    def forward(self, *args, **kwargs):
        raise NotImplementedError()


def _test_ex2():
    # --- invariant 1: fresh module starts in training mode ---
    class TinyLayer(Module):
        def __init__(self, in_f, out_f):
            super().__init__()
            self.weight = Parameter(t.zeros(in_f, out_f))
            self.bias = Parameter(t.zeros(out_f))

    class Net(Module):
        def __init__(self):
            super().__init__()
            self.layer1 = TinyLayer(3, 5)
            self.layer2 = TinyLayer(5, 2)

    net = Net()
    assert net.training is True, 'fresh module must start in training mode'
    assert net.layer1.training is True and net.layer2.training is True

    # --- invariant 2: eval() propagates to every submodule ---
    ret = net.eval()
    assert ret is net, 'eval() must return self for chaining'
    assert net.training is False
    assert net.layer1.training is False, 'eval must propagate to layer1'
    assert net.layer2.training is False, 'eval must propagate to layer2'

    # --- invariant 3: train() restores all submodules ---
    ret2 = net.train()
    assert ret2 is net, 'train() must return self for chaining'
    assert net.training is True
    assert net.layer1.training is True and net.layer2.training is True

    # --- invariant 4: deeper nesting — three levels ---
    class Block(Module):
        def __init__(self):
            super().__init__()
            self.inner = TinyLayer(4, 4)

    class Stack(Module):
        def __init__(self):
            super().__init__()
            self.b1 = Block()
            self.b2 = Block()

    stack = Stack()
    stack.eval()
    # Every leaf must have training=False, across 3 levels.
    assert stack.training is False
    assert stack.b1.training is False and stack.b2.training is False
    assert stack.b1.inner.training is False, 'must reach 3-deep nested leaf'
    assert stack.b2.inner.training is False

    # --- invariant 5: train(False) is alias for eval() ---
    stack.train(True)
    assert stack.training is True and stack.b1.inner.training is True
    stack.train(False)
    assert stack.training is False and stack.b1.inner.training is False
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
class Parameter(MiniTensor):
    def __init__(self, array, requires_grad: bool = True):
        super().__init__(array, requires_grad=requires_grad)


class Module:
    def __init__(self):
        object.__setattr__(self, '_parameters', {})
        object.__setattr__(self, '_modules', {})
        object.__setattr__(self, 'training', True)

    def __setattr__(self, name, value):
        if isinstance(value, Parameter):
            self._parameters[name] = value
            self._modules.pop(name, None)
        elif isinstance(value, Module):
            self._modules[name] = value
            self._parameters.pop(name, None)
        else:
            self._parameters.pop(name, None)
            self._modules.pop(name, None)
        object.__setattr__(self, name, value)

    def parameters(self):
        for p in self._parameters.values():
            yield p
        for m in self._modules.values():
            yield from m.parameters()

    def train(self, mode: bool = True):
        self.training = mode
        for m in self._modules.values():
            m.train(mode)
        return self

    def eval(self):
        return self.train(False)

    def forward(self, *args, **kwargs):
        raise NotImplementedError()
```

**Why `eval` is a thin wrapper.** Keeping `eval()` as `self.train(False)` (rather than duplicating the recursion) means any future logic added to `train` — logging, hook firing, switching cuDNN deterministic flags — automatically applies to `eval`. PyTorch's actual `nn.Module.eval` is one line for this reason.

**Bootstrap order is load-bearing.** Setting `training = True` in `__init__` MUST use `object.__setattr__`. The custom `__setattr__` would otherwise check `isinstance(value, Parameter)` / `isinstance(value, Module)` — fine here — but the BOOLEAN True falls to the else branch which calls `self._parameters.pop(name, None)`. If `_parameters` was already installed it's fine; if not, AttributeError. Always bootstrap with `object.__setattr__`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()